# 01 - Galactic dust maps (SFD 1998)

- author : Sylvie Dagoret-Campagne
- creation date : 2026-09-19
- kernel : conda_py313_opsim53

Inspection of the Galactic dust maps stored in `maps/DustMaps`:

* `dust_nside_<N>.npz` - HEALPix representations of the SFD E(B-V) map for `NSIDE` = 2 ... 1024
  (key `ebvMap`, **RING** ordering, pixel centres defined on **RA/Dec**), as described in the `README`;
* `SFD_dust_4096_ngp.fits` / `SFD_dust_4096_sgp.fits` - the original SFD maps, in Lambert (zenithal equal-area)
  projection around the North / South Galactic poles, 4096 x 4096 pixels.

Reference: Schlegel, Finkbeiner & Davis (1998), [doi:10.1086/305772](https://iopscience.iop.org/article/10.1086/305772).

**Contents**
1. Inventory of the available maps
2. Basic statistics
3. Full-sky maps (equatorial and Galactic coordinates)
4. Views towards the Galactic centre and the poles
5. Distribution of E(B-V) and cumulative sky fraction
6. E(B-V) as a function of Galactic latitude
7. Effect of the HEALPix resolution
8. Original SFD Lambert maps (FITS) and cross-check with the HEALPix maps

In [ ]:
from pathlib import Path
import re

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import healpy as hp
from astropy.io import fits
from scipy.ndimage import map_coordinates

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

# The notebook lives in notebooks/09_RubinMaps/ and the maps were copied to ./maps
CANDIDATES = [
    Path("maps/DustMaps"),
    Path.home() / "DATA/OpSim/maps/DustMaps",
]
DATA_DIR = next(p for p in CANDIDATES if p.is_dir())

# Common colour scale for E(B-V) [mag]
VMIN, VMAX = 0.01, 1.0

## 1. Inventory of the available maps

In [ ]:
def nside_of(path):
    return int(re.search(r"dust_nside_(\d+)\.npz", path.name).group(1))


AVAILABLE = sorted(nside_of(p) for p in DATA_DIR.glob("dust_nside_*.npz"))


def load_ebv(nside):
    """Return the E(B-V) HEALPix map (RING ordering, pixel centres defined on RA/Dec)."""
    with np.load(DATA_DIR / f"dust_nside_{nside}.npz") as f:
        return f["ebvMap"]


print(f"Data directory: {DATA_DIR.resolve()}\n")
print(f'{"nside":>6} {"npix":>10} {"pixel size":>14} {"file size":>11}')
for ns in AVAILABLE:
    size_mb = (DATA_DIR / f"dust_nside_{ns}.npz").stat().st_size / 1e6
    print(f"{ns:6d} {hp.nside2npix(ns):10d} {hp.nside2resol(ns, arcmin=True):9.1f} arcmin {size_mb:8.2f} MB")

## 2. Basic statistics

`NSIDE_VIEW` sets the resolution used for the full-sky figures (256 is a good compromise between
detail and speed; 1024 is available for finer views).

In [ ]:
NSIDE_VIEW = 256 if 256 in AVAILABLE else max(AVAILABLE)
ebv = load_ebv(NSIDE_VIEW)
assert ebv.size == hp.nside2npix(NSIDE_VIEW)

pcts = [1, 5, 25, 50, 75, 95, 99, 99.9]
print(f"NSIDE = {NSIDE_VIEW}: {ebv.size} pixels, dtype = {ebv.dtype}")
print(f"  min / max             : {ebv.min():.4f} / {ebv.max():.2f} mag")
print(f"  mean / median         : {ebv.mean():.4f} / {np.median(ebv):.4f} mag")
print(
    "  percentiles           :", ", ".join(f"{p}% = {v:.3f}" for p, v in zip(pcts, np.percentile(ebv, pcts)))
)
print(f"  non-finite pixels     : {(~np.isfinite(ebv)).sum()}")

In [ ]:
def pixel_lb(nside):
    """Galactic (l, b) in degrees of the pixel centres (the maps are defined on RA/Dec pixel centres)."""
    ra, dec = hp.pix2ang(nside, np.arange(hp.nside2npix(nside)), lonlat=True)
    l, b = hp.Rotator(coord=["C", "G"])(ra, dec, lonlat=True)
    return l, b


l_pix, b_pix = pixel_lb(NSIDE_VIEW)

## 3. Full-sky maps

The HEALPix grid of these maps is defined in equatorial coordinates (left); rotating to Galactic
coordinates (right) puts the Galactic plane along the equator of the map. Logarithmic colour scale.

In [ ]:
fig = plt.figure(figsize=(14, 5.5))
hp.mollview(
    ebv,
    coord="C",
    norm="log",
    min=VMIN,
    max=VMAX,
    unit="E(B-V) [mag]",
    title=f"SFD E(B-V) - equatorial (RA/Dec), NSIDE={NSIDE_VIEW}",
    sub=(1, 2, 1),
    fig=fig.number,
)
hp.graticule()
hp.mollview(
    ebv,
    coord=["C", "G"],
    norm="log",
    min=VMIN,
    max=VMAX,
    unit="E(B-V) [mag]",
    title=f"SFD E(B-V) - Galactic (l, b), NSIDE={NSIDE_VIEW}",
    sub=(1, 2, 2),
    fig=fig.number,
)
hp.graticule()
plt.show()

In [ ]:
# Linear stretch, saturated at E(B-V) = 0.5 mag: highlights the extent of the "clean" high-latitude sky
hp.mollview(
    ebv,
    coord=["C", "G"],
    min=0,
    max=0.5,
    unit="E(B-V) [mag]",
    cmap="viridis",
    title="SFD E(B-V) - Galactic, linear stretch clipped at 0.5 mag",
)
hp.graticule()
plt.show()

## 4. Views towards the Galactic centre and the poles

In [ ]:
views = {
    "Galactic centre (l=0, b=0)": (0, 0),
    "North Galactic Pole": (0, 90),
    "South Galactic Pole": (0, -90),
}

fig = plt.figure(figsize=(15, 5))
for i, (name, rot) in enumerate(views.items(), start=1):
    hp.orthview(
        ebv,
        coord=["C", "G"],
        rot=rot,
        half_sky=True,
        norm="log",
        min=VMIN,
        max=VMAX,
        unit="E(B-V) [mag]",
        title=name,
        sub=(1, 3, i),
        fig=fig.number,
    )
    hp.graticule()
plt.show()

## 5. Distribution of E(B-V) and cumulative sky fraction

All HEALPix pixels have the same area, so the pixel counts are directly proportional to sky area.

In [ ]:
srt = np.sort(ebv)
pix_area = hp.nside2pixarea(NSIDE_VIEW, degrees=True)  # deg^2
thresholds = (0.05, 0.1, 0.2, 0.3, 0.5, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

bins = np.logspace(np.log10(ebv.min()), np.log10(ebv.max()), 120)
axes[0].hist(ebv, bins=bins, alpha=0.75)
axes[0].axvline(np.median(ebv), color="k", ls="--", label=f"median = {np.median(ebv):.3f}")
axes[0].set(xscale="log", yscale="log", xlabel="E(B-V) [mag]", ylabel="number of pixels", title="Histogram")
axes[0].legend()

axes[1].plot(srt, np.arange(1, srt.size + 1) / srt.size)
for thr in thresholds:
    axes[1].axvline(thr, color="gray", lw=0.6, ls=":")
axes[1].set(
    xscale="log",
    xlabel="E(B-V) threshold [mag]",
    ylabel="sky fraction with E(B-V) < threshold",
    title="Cumulative distribution",
    ylim=(0, 1.02),
)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'{"E(B-V) <":>10} {"sky fraction":>13} {"area [deg2]":>13}')
for thr in thresholds:
    n_below = np.searchsorted(srt, thr)
    print(f"{thr:10.2f} {n_below / srt.size:13.1%} {n_below * pix_area:13.0f}")

## 6. E(B-V) as a function of Galactic latitude

Median and 16-84% range of E(B-V) in 5-degree bins of Galactic latitude, showing the
north/south asymmetry and the steep rise towards the Galactic plane.

In [ ]:
edges = np.arange(-90, 91, 5)
centres = 0.5 * (edges[1:] + edges[:-1])
idx = np.digitize(b_pix, edges) - 1

med, lo, hi = np.full((3, centres.size), np.nan)
for k in range(centres.size):
    sel = ebv[idx == k]
    if sel.size:
        lo[k], med[k], hi[k] = np.percentile(sel, [16, 50, 84])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.fill_between(centres, lo, hi, alpha=0.3, label="16-84 %")
ax.plot(centres, med, "o-", label="median")
ax.set(yscale="log", xlabel="Galactic latitude b [deg]", ylabel="E(B-V) [mag]", xlim=(-90, 90))
ax.grid(alpha=0.3, which="both")
ax.legend()
plt.show()

## 7. Effect of the HEALPix resolution

Same Galactic field seen at different `NSIDE`, followed by a consistency check between the
highest-resolution map degraded to a lower `NSIDE` (mean of the sub-pixels) and the native
lower-resolution map. This tells us whether the coarse maps were obtained by point-sampling
SFD at the pixel centres or by averaging.

In [ ]:
FIELD_LB = (0.0, -15.0)  # (l, b) of the field centre [deg]
FOV_DEG = 30.0  # field of view [deg]
XSIZE = 500

nsides_cmp = [n for n in (16, 64, 256, 1024) if n in AVAILABLE]
fig = plt.figure(figsize=(12, 10))
for i, ns in enumerate(nsides_cmp, start=1):
    m = ebv if ns == NSIDE_VIEW else load_ebv(ns)
    hp.gnomview(
        m,
        coord=["C", "G"],
        rot=FIELD_LB,
        xsize=XSIZE,
        reso=FOV_DEG * 60 / XSIZE,
        norm="log",
        min=VMIN,
        max=VMAX,
        unit="E(B-V) [mag]",
        title=f"NSIDE = {ns}  ({hp.nside2resol(ns, arcmin=True):.1f} arcmin)",
        sub=(2, 2, i),
        fig=fig.number,
    )
    hp.graticule()
plt.show()

In [ ]:
NS_LO = 64
NS_HI = max(AVAILABLE)

if NS_HI > NS_LO and NS_LO in AVAILABLE:
    lo_native = load_ebv(NS_LO)
    hi_degraded = hp.ud_grade(load_ebv(NS_HI), NS_LO, order_in="RING", order_out="RING")
    ratio = lo_native / hi_degraded

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    hb = axes[0].hexbin(
        hi_degraded, lo_native, xscale="log", yscale="log", bins="log", gridsize=120, mincnt=1
    )
    lim = [min(lo_native.min(), hi_degraded.min()), max(lo_native.max(), hi_degraded.max())]
    axes[0].plot(lim, lim, "r-", lw=0.8)
    axes[0].set(xlabel=f"NSIDE={NS_HI} degraded to {NS_LO} [mag]", ylabel=f"native NSIDE={NS_LO} [mag]")
    fig.colorbar(hb, ax=axes[0], label="pixels")

    axes[1].hist(ratio, bins=np.linspace(0.5, 1.5, 150))
    axes[1].set(xlabel="native / degraded", ylabel="number of pixels")
    plt.tight_layout()
    plt.show()

    p16, p50, p84 = np.percentile(ratio, [16, 50, 84])
    print(f"native / degraded ratio: median = {p50:.3f}, 68% interval = [{p16:.3f}, {p84:.3f}]")
else:
    print("Need NSIDE=64 and a higher-resolution map to run this check.")

## 8. Original SFD Lambert maps (FITS)

Two 4096 x 4096 images: north (`b >= 0`) and south (`b < 0`) Galactic hemispheres in Lambert
equal-area projection, centred on the respective pole with the Galactic equator on the circle
inscribed in the image.

In [ ]:
sfd = {}
for pole in ("ngp", "sgp"):
    path = DATA_DIR / f"SFD_dust_4096_{pole}.fits"
    if path.exists():
        with fits.open(path) as hdul:
            sfd[pole] = {"data": np.array(hdul[0].data, dtype=np.float32), "header": hdul[0].header.copy()}

for pole, d in sfd.items():
    print(f'--- {pole.upper()}: shape = {d["data"].shape}, dtype = {d["data"].dtype}')
    print(f"    min / max / median (finite, > 0): ", end="")
    v = d["data"][np.isfinite(d["data"]) & (d["data"] > 0)]
    print(f"{v.min():.4f} / {v.max():.2f} / {np.median(v):.4f}")

if "ngp" in sfd:
    print()
    print(repr(sfd["ngp"]["header"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for ax, pole, title in zip(
    axes, ("ngp", "sgp"), ("North Galactic hemisphere (b >= 0)", "South Galactic hemisphere (b < 0)")
):
    if pole not in sfd:
        ax.set_visible(False)
        continue
    img = np.where(sfd[pole]["data"] > 0, sfd[pole]["data"], np.nan)
    im = ax.imshow(img, origin="lower", cmap="viridis", norm=LogNorm(vmin=VMIN, vmax=VMAX))
    ax.set(title=f"SFD E(B-V) - {title}", xlabel="x [pixel]", ylabel="y [pixel]")
    fig.colorbar(im, ax=ax, shrink=0.8, label="E(B-V) [mag]")
plt.tight_layout()
plt.show()

### Cross-check: SFD Lambert maps vs HEALPix map

The Lambert maps are sampled (bilinear interpolation) at the Galactic coordinates of the HEALPix
pixel centres and compared with the HEALPix map. With the standard SFD Lambert convention

* `x = (N/2) sqrt(1 - n sin b) cos l + N/2 - 1/2`
* `y = -n (N/2) sqrt(1 - n sin b) sin l + N/2 - 1/2`

(`N` = number of pixels per side, `n = +1` for the NGP map and `-1` for the SGP map).
If the median ratio below is not close to 1, check this convention against the FITS header printed above.

In [ ]:
def lambert_xy(l_deg, b_deg, n, npix):
    """SFD Lambert projection: Galactic (l, b) in degrees -> pixel (x, y) (0-based), n = +1 (NGP) or -1 (SGP)."""
    l, b = np.radians(l_deg), np.radians(b_deg)
    r = np.sqrt(1.0 - n * np.sin(b))
    x = 0.5 * npix * r * np.cos(l) + 0.5 * npix - 0.5
    y = -n * 0.5 * npix * r * np.sin(l) + 0.5 * npix - 0.5
    return x, y


def sample_sfd(l_deg, b_deg):
    """Bilinear interpolation of the SFD Lambert maps at Galactic (l, b) [deg]."""
    out = np.full(l_deg.shape, np.nan)
    for pole, n, sel in (("ngp", +1, b_deg >= 0), ("sgp", -1, b_deg < 0)):
        npix = sfd[pole]["data"].shape[0]
        x, y = lambert_xy(l_deg[sel], b_deg[sel], n, npix)
        out[sel] = map_coordinates(sfd[pole]["data"], [y, x], order=1, mode="nearest")
    return out


if {"ngp", "sgp"} <= sfd.keys():
    ebv_sfd = sample_sfd(l_pix, b_pix)
    ratio = ebv / ebv_sfd
    good = np.isfinite(ratio) & (ebv_sfd > 0)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    hb = axes[0].hexbin(
        ebv_sfd[good], ebv[good], xscale="log", yscale="log", bins="log", gridsize=120, mincnt=1
    )
    lim = [ebv[good].min(), ebv[good].max()]
    axes[0].plot(lim, lim, "r-", lw=0.8)
    axes[0].set(xlabel="SFD Lambert map, sampled [mag]", ylabel=f"HEALPix NSIDE={NSIDE_VIEW} [mag]")
    fig.colorbar(hb, ax=axes[0], label="pixels")
    axes[1].hist(ratio[good], bins=np.linspace(0.5, 1.5, 150))
    axes[1].set(xlabel="HEALPix / SFD Lambert", ylabel="number of pixels")
    plt.tight_layout()
    plt.show()

    p16, p50, p84 = np.percentile(ratio[good], [16, 50, 84])
    print(
        f"HEALPix / SFD ratio: median = {p50:.3f}, 68% interval = [{p16:.3f}, {p84:.3f}]  ({good.sum()} pixels)"
    )
else:
    print("SFD FITS files not found: skipping the cross-check.")

## Notes

* The SFD E(B-V) values are known to be overestimated by about 14% with respect to the recalibration of
  Schlafly & Finkbeiner (2011): multiply by 0.86 when converting to extinction with their coefficients.
* SFD is unreliable at very low Galactic latitude (very high E(B-V)) and in regions with strong
  contamination of the far-infrared emission (e.g. the Magellanic Clouds).
* Next steps for this folder: other maps in `maps/` (to be examined in the following notebooks).